# 第8回 課題：MLOps実践（穴埋め問題）

本日学んだ内容を定着させる課題です。`# TODO` の箇所を埋めて、各セル・記述問題に取り組んでください。

解答後、`exercise_answer.ipynb` で答え合わせができます。

## 問1. Dockerfile の主要命令

Python 3.11 ベースで、`requirements.txt` の依存をインストールし、`main.py` を `uvicorn` で起動する Dockerfile を作ります。**レイヤキャッシュを意識した順序**にしてください。

```dockerfile
# ベースイメージ
____ python:3.11-slim

WORKDIR /app

# 依存ライブラリの定義だけを先にコピー
____ requirements.txt .
____ pip install --no-cache-dir -r requirements.txt

# アプリ本体は依存より後にコピー
____ main.py .

EXPOSE 8000

# コンテナ起動コマンド
____ ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

下のセルに、5つの `____` を埋めた**完全な Dockerfile** を文字列として記入してください（読みやすさのため triple-quoted で OK）。


In [ ]:
answer_dockerfile = '''
（ここに穴埋めを完成させた Dockerfile を記述）
'''
print(answer_dockerfile)

## 問2. `docker run` のポートフォワード

ビルド済みの `mnist-api:latest` を、**ホストの 9000 番**にコンテナの 8000 番を繋いだ状態で、バックグラウンド起動するコマンドを書きます。

```bash
# ホストのポート9000を、コンテナの8000に繋ぐ
docker run ____ --rm --name mnist-api ____ ____ mnist-api:latest
```

下のセルに、3つの `____` を埋めた**完全な docker run コマンド**を文字列として記入してください。

In [ ]:
answer_docker_run = "（ここに穴埋めを完成させた docker run コマンドを記述）"
print(answer_docker_run)

## 問3: GitHub Actions ワークフロー

次のワークフローを完成させよ。要件：
- main ブランチへの push でトリガ
- ubuntu の最新 runner で実行
- pytest を回す

In [ ]:
workflow = '''
name: CI

___TODO_6___:                          # ワークフローのトリガ指定（push などをぶら下げる親キー）
  push:
    branches: ["main"]

jobs:
  test:
    runs-on: ___TODO_7___              # 最新の Ubuntu runner
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install
        run: pip install -r app/requirements.txt pytest httpx
      - name: Test
        ___TODO_8___: pytest app/tests -v   # シェルコマンドを書くキー（uses ではなく？）
'''
print(workflow)

## 問4: K8s Deployment マニフェスト

次のマニフェストの空欄を埋めよ。要件：
- Pod を 3 つ常時動かしたい
- イメージは `mnist-app:v2` を使う（ローカルにビルド済み）
- コンテナの待ち受けポートは 8000

In [ ]:
manifest = '''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mnist-app
spec:
  replicas: ___TODO_3___      # 何個動かすか
  selector:
    matchLabels:
      app: mnist-app
  template:
    metadata:
      labels:
        app: mnist-app
    spec:
      containers:
        - name: mnist-app
          image: ___TODO_4___          # ローカルでビルドしたイメージ名:タグ
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: ___TODO_5___    # main.py の uvicorn が listen するポート
'''
print(manifest)

## 問5: PSI（Population Stability Index）の計算

PSI の式：
$$\mathrm{PSI} = \sum_i (p_i^{\text{cur}} - p_i^{\text{ref}}) \cdot \ln \frac{p_i^{\text{cur}}}{p_i^{\text{ref}}}$$

TODO を埋めて関数を完成させよ。

In [ ]:
import numpy as np

def calc_psi(reference, current, bins=10, eps=1e-6):
    quantiles = np.linspace(0, 1, bins + 1)
    bin_edges = np.quantile(reference, quantiles)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    ref_counts, _ = np.histogram(reference, bins=bin_edges)
    cur_counts, _ = np.histogram(current, bins=bin_edges)

    p_ref = ref_counts / ref_counts.sum() + eps
    p_cur = cur_counts / cur_counts.sum() + eps

    # TODO 9: 上の式に従って PSI を計算する1行を書け
    psi = ___TODO_9___
    return float(psi)

rng = np.random.default_rng(0)
ref = rng.normal(50, 10, 5000)
cur = rng.normal(55, 10, 5000)
print(f'PSI = {calc_psi(ref, cur):.4f}')
# 期待値: 0.1〜0.3 程度

## 問6: しきい値判定ロジック

PSI のしきい値を以下のように判定する関数を完成させよ：
- 0.1 未満 → 'stable'
- 0.1 以上 0.2 未満 → 'warning'
- 0.2 以上 → 'drift'

In [ ]:
def judge(psi: float) -> str:
    # TODO 10: 上記ルールに沿って分岐を書く
    pass

for v in [0.05, 0.15, 0.30]:
    print(v, '->', judge(v))

## 問7: 記述問題

以下にあなたの言葉で答えよ（解答例は exercise_answer.ipynb 参照）。

### Q7-1
**コードを変えていないのにモデルの精度が落ちる現象を何と呼ぶか。検知手法を1つ挙げ、その手法が何を判定しているかを述べよ。**

（ここに記述）

### Q7-2
**データドリフトを検知してから再学習までの実運用フローを4ステップで述べよ。**

（ここに記述）

### Q7-3
**`docker run -p 8000:8000 mnist-app:v2` で十分に運用できないのはなぜか。Kubernetes が解決する問題を3つ挙げよ。**

（ここに記述）